In [5]:
import pandas as pd
from pathlib import Path
from dotenv import find_dotenv

# Âncora da raiz do projeto (usecwd=True: robusto em notebook; chamado 1x).
# Exploração é local — não carrega credenciais.
DOTENV = find_dotenv(usecwd=True)
assert DOTENV, ".env não encontrado — abra o notebook de dentro do projeto"
PROJECT_ROOT  = Path(DOTENV).parent
EXTRAIDOS_DIR = PROJECT_ROOT / "data_lake" / "external" / "extraidos"

# Padrão dos CSVs do INEP: separador ';' e encoding latin1.
PARAMS_LEITURA = {
    'sep': ';',
    'encoding': 'ISO-8859-1',
    'low_memory': False,   # evita aviso de tipos mistos na leitura completa
}
print("Projeto:", PROJECT_ROOT)

Projeto: /mnt/d/diego/01_projects/postech-challenge-2


In [6]:
caminho_municipio = EXTRAIDOS_DIR / "2023" / "TS_MUNICIPIO.csv"
df_mun = pd.read_csv(caminho_municipio, **PARAMS_LEITURA)

print(f"Linhas : {len(df_mun):,}")
print(f"Colunas: {df_mun.shape[1]}")
print("\n--- info(): schema, não-nulos e memória ---")
df_mun.info()
print("\n--- Primeiras 5 linhas ---")
display(df_mun.head())

Linhas : 11,547
Colunas: 9

--- info(): schema, não-nulos e memória ---
<class 'pandas.DataFrame'>
RangeIndex: 11547 entries, 0 to 11546
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NU_ANO_AVALIACAO       11547 non-null  int64  
 1   CO_UF                  11547 non-null  int64  
 2   SG_UF                  11547 non-null  str    
 3   CO_MUNICIPIO           11547 non-null  int64  
 4   NO_MUNICIPIO           11547 non-null  str    
 5   TP_SERIE               11547 non-null  int64  
 6   ID_TIPO_REDE           11547 non-null  int64  
 7   PC_ALUNO_ALFABETIZADO  11547 non-null  float64
 8   VL_MEDIA_LP            11547 non-null  float64
dtypes: float64(2), int64(5), str(2)
memory usage: 971.1 KB

--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304
1,2023,11,RO,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304
2,2023,11,RO,1100023,Ariquemes,2,3,62.30,757.0999
3,2023,11,RO,1100023,Ariquemes,2,5,62.30,757.0999
4,2023,11,RO,1100031,Cabixi,2,5,69.10,767.8763


In [7]:
caminho_aluno = EXTRAIDOS_DIR / "2023" / "TS_ALUNO.csv"

# Leitura COMPLETA (~1,7M linhas). pandas é eager: ler já materializa tudo,
# não há "count preguiçoso" como no Spark. Roda em segundos e cabe na RAM.
# Para só espiar rápido: pd.read_csv(caminho_aluno, nrows=5000, **PARAMS_LEITURA)
df_aluno = pd.read_csv(caminho_aluno, **PARAMS_LEITURA)

print(f"Linhas : {len(df_aluno):,}")
print(f"Colunas: {df_aluno.shape[1]}")
# info() ganha o jogo AQUI: os não-nulos denunciam as colunas de proficiência
# vazias dos alunos AUSENTES (IN_PRESENCA_LP=0) — a base do filtro da Gold.
print("\n--- info(): repare a diferença de não-nulos entre colunas ---")
df_aluno.info()
print("\n--- Primeiras 5 linhas ---")
display(df_aluno.head())

Linhas : 1,747,439
Colunas: 15

--- info(): repare a diferença de não-nulos entre colunas ---
<class 'pandas.DataFrame'>
RangeIndex: 1747439 entries, 0 to 1747438
Data columns (total 15 columns):
 #   Column               Dtype  
---  ------               -----  
 0   NU_ANO_AVALIACAO     int64  
 1   CO_UF                int64  
 2   SG_UF                str    
 3   ID_ALUNO             int64  
 4   TP_SERIE             int64  
 5   ID_ESCOLA            int64  
 6   TP_DEPENDENCIA       int64  
 7   CO_MUNICIPIO         int64  
 8   NO_MUNICIPIO         str    
 9   IN_PRESENCA_LP       int64  
 10  IN_PREENCHIMENTO_LP  int64  
 11  CO_CADERNO_LP        int64  
 12  VL_PESO_ALUNO_LP     float64
 13  VL_PROFICIENCIA_LP   float64
 14  IN_ALFABETIZADO      int64  
dtypes: float64(2), int64(11), str(2)
memory usage: 221.8 MB

--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0


In [8]:
caminho_estado = EXTRAIDOS_DIR / "2023" / "TS_ESTADO.csv"
df_est = pd.read_csv(caminho_estado, **PARAMS_LEITURA)

print(f"Linhas : {len(df_est)}")
print(f"Colunas: {df_est.shape[1]}")

# describe() só nas MEDIDAS reais. CO_UF, TP_SERIE, ID_TIPO_REDE são códigos:
# média e desvio deles não têm significado, apesar de o describe() calcular.
COLS_MEDIDA = ['PC_ALUNO_ALFABETIZADO', 'VL_MEDIA_LP']
print("\n--- Resumo estatístico (só medidas) ---")
display(df_est[COLS_MEDIDA].describe())
print("\n--- Primeiras 5 linhas ---")
display(df_est.head())

Linhas : 70
Colunas: 7

--- Resumo estatístico (só medidas) ---


,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
count,70.000000,70.000000
mean,55.319000,744.314474
std,13.059867,16.147667
min,30.570000,712.562000
25%,44.960000,733.381700
50%,54.415000,744.668700
75%,63.520000,751.650700
max,84.490000,795.729300



--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
0,2023,11,RO,2,2,58.65,751.4731
1,2023,11,RO,2,3,65.17,760.1971
2,2023,11,RO,2,5,64.60,759.4357
3,2023,13,AM,2,3,49.20,733.6637
4,2023,13,AM,2,5,52.20,736.4687


In [15]:
caminho_xlsx = EXTRAIDOS_DIR / "2023" / "resultados_e_metas_municipios.xlsx"

# header=1 (base 0): a planilha tem DUAS linhas de cabeçalho.
#   linha 1 = rótulos verbosos ("CÓDIGO MUNICÍPIO")
#   linha 2 = códigos técnicos ("CO_MUNICIPIO") -> mesma convenção dos CSVs
df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# Rodapé: 1 linha em branco + 2 "Observação" no fim. Corta filtrando chave nula.
df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

print(f"Linhas : {len(df_mun_meta):,}")     # 5.468
print(f"Colunas: {df_mun_meta.shape[1]}")   # 16
print("\nColunas:", list(df_mun_meta.columns))
display(df_mun_meta.head())

Linhas : 5,468
Colunas: 16

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'NIVEIS_ALFABETIZACAO_2023', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,NIVEIS_ALFABETIZACAO_2023,PC_AVALIADOS_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,64.55,67.07860126116485,69.5120284348663,71.84109325785958,74.05863395650297,76.159493418938,78.14043382279495,80,3,89.37
1,2023,11,RO,1100023,Ariquemes,MUNICIPAL,62.3,65.21687842070342,68.02390876920695,70.7061609144921,73.25188946172531,75.65256653626597,77.90277676917654,80,3,89.79
2,2023,11,RO,1100031,Cabixi,MUNICIPAL,69.1,70.84502912613192,72.53068615333729,74.15443845377203,75.71434639019579,77.20904206573897,78.63769980684307,80,3,90.48
3,2023,11,RO,1100049,Cacoal,MUNICIPAL,62.51,65.39071596861464,68.16281698785075,70.81199011575661,73.32698568453318,75.69964198045608,77.92478110638412,80,3,84.44
4,2023,11,RO,1100056,Cerejeiras,MUNICIPAL,58.53,62.09040103983767,65.52517196826541,68.80509082659395,71.9067479925146,74.81290415754658,77.51244530506875,80,2,92.12


In [12]:
dic = pd.read_excel(
    caminho_xlsx, sheet_name="variáveis", header=None,
    usecols=[1, 2], names=["campo", "descricao"]
).dropna(how="all")
display(dic)

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS (1),Percentual de alunos alfabetizados: percentual...
8,META 2024 (2),Meta estabelecida para o município em 2024
9,META 2025,Meta estabelecida para o município em 2025
10,META 2026,Meta estabelecida para o município em 2026


In [17]:
caminho_xlsx = EXTRAIDOS_DIR / "2023" / "resultados_e_metas_ufs.xlsx"

# A planilha também possui duas linhas de cabeçalho.
# header=1 utiliza a segunda linha, que contém os códigos técnicos.
df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# Remove linhas do rodapé (observações)
df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")
print("\nColunas:", list(df_uf_meta.columns))

display(df_uf_meta.head())

Linhas : 27
Colunas: 16

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'SAEB_2019', 'SAEB_2021', 'PC_ALUNO_ALFABETIZADO', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,SAEB_2019,SAEB_2021,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2023,12,AC,Acre,PÚBLICA,52.87,20.05,-,-,-,-,-,-,-,-,-
2,2023,27,AL,Alagoas,PÚBLICA,39.01,30.04,43.88,49.699999999999996,55.5,61.1,66.5,71.5,76,> 80,92.36
3,2023,13,AM,Amazonas,PÚBLICA,43.83,28.76,52.2,56.8,61.3,65.6,69.6,73.4,76.9,> 80,76.14
4,2023,16,AP,Amapá,PÚBLICA,24.77,18.77,41.56,47.599999999999994,53.8,59.900000000000006,65.6,70.9,75.8,> 80,89.68
5,2023,29,BA,Bahia,PÚBLICA,41.36,24.49,36.8,43.4,50.199999999999996,57.1,63.6,69.80000000000001,75.19999999999999,> 80,84.53


In [19]:
dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

display(dic_uf)

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,META 2024 (2),Meta estabelecida para o estado ou Brasil em 2024
9,META 2025,Meta estabelecida para o estado ou Brasil em 2025


In [25]:
caminho_xlsx = EXTRAIDOS_DIR / "2024" / "resultados_e_metas_municipios_2024.xlsx"

# header=1 (base 0): a planilha tem DUAS linhas de cabeçalho.
#   linha 1 = rótulos verbosos ("CÓDIGO MUNICÍPIO")
#   linha 2 = códigos técnicos ("CO_MUNICIPIO") -> mesma convenção dos CSVs
df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# Rodapé: 1 linha em branco + 2 "Observação" no fim. Corta filtrando chave nula.
df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

print(f"Linhas : {len(df_mun_meta):,}")     # 5.468
print(f"Colunas: {df_mun_meta.shape[1]}")   # 16
print("\nColunas:", list(df_mun_meta.columns))
display(df_mun_meta.head())

Linhas : 5,352
Colunas: 17

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'CO_NIVEL_ALFABETIZACAO', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,CO_NIVEL_ALFABETIZACAO,PC_AVALIADOS_LP
0,2024,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,64.6,67.79,67.08,69.51,71.84,74.06,76.16,78.14,80,3,89.86928104575163
1,2024,11,RO,1100023,Ariquemes,MUNICIPAL,62.3,65.62,65.22,68.02,70.71,73.25,75.65,77.9,80,3,88.759367194005
2,2024,11,RO,1100031,Cabixi,MUNICIPAL,69.1,75.88,70.85,72.53,74.15,75.71,77.21,78.64,80,4,92.5925925925926
3,2024,11,RO,1100049,Cacoal,MUNICIPAL,62.5,65.81,65.39,68.16,70.81,73.33,75.7,77.92,80,3,92.56678281068524
4,2024,11,RO,1100056,Cerejeiras,MUNICIPAL,58.5,66.81,62.09,65.53,68.81,71.91,74.81,77.51,80,3,96.44268774703558


In [27]:
dic = pd.read_excel(
    caminho_xlsx, sheet_name="Variáveis", header=None,
    usecols=[1, 2], names=["campo", "descricao"]
).dropna(how="all")
display(dic)

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2023 (1),Percentual de alunos alfabetizados: percentual...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2024 (1),Percentual de alunos alfabetizados: percentual...
9,META 2024 (2),Meta estabelecida para o município em 2024
10,META 2025,Meta estabelecida para o município em 2025


In [34]:
caminho_xlsx = EXTRAIDOS_DIR / "2024" / "resultados_e_metas_ufs_2024_2.xlsx"

# A planilha também possui duas linhas de cabeçalho.
# header=1 utiliza a segunda linha, que contém os códigos técnicos.
df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# Remove linhas do rodapé (observações)
df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")
print("\nColunas:", list(df_uf_meta.columns))

display(df_uf_meta.head())

Linhas : 27
Colunas: 15

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2024,12,AC,Acre,PÚBLICA,-,51.38,-,56.9,62.2,67.3,72,76.2,> 80,80.86922165152114
2,2024,27,AL,Alagoas,PÚBLICA,43.88,48.63,49.7,55.5,61.1,66.5,71.5,76,> 80,93.78396840010691
3,2024,13,AM,Amazonas,PÚBLICA,52.2,49.17,56.8,61.3,65.6,69.6,73.4,76.9,> 80,79.49477411227998
4,2024,16,AP,Amapá,PÚBLICA,41.56,46.62,47.6,53.8,59.9,65.6,70.9,75.8,> 80,89.14149443561207
5,2024,29,BA,Bahia,PÚBLICA,36.8,35.96,43.4,50.2,57.1,63.6,69.8,75.2,> 80,90.04421141883181


In [35]:
dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

display(dic_uf)

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
9,META 2024,Meta estabelecida para o estado ou Brasil em 2024


In [29]:
caminho_xlsx = EXTRAIDOS_DIR / "2025" / "resultados_e_metas_municipios_2025_v2.xlsx"

# header=1 (base 0): a planilha tem DUAS linhas de cabeçalho.
#   linha 1 = rótulos verbosos ("CÓDIGO MUNICÍPIO")
#   linha 2 = códigos técnicos ("CO_MUNICIPIO") -> mesma convenção dos CSVs
df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# Rodapé: 1 linha em branco + 2 "Observação" no fim. Corta filtrando chave nula.
df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

print(f"Linhas : {len(df_mun_meta):,}")     # 5.468
print(f"Colunas: {df_mun_meta.shape[1]}")   # 16
print("\nColunas:", list(df_mun_meta.columns))
display(df_mun_meta.head())

Linhas : 5,466
Colunas: 18

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'PC_ALUNO_ALFABETIZADO_2025', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'CO_NIVEL_ALFABETIZACAO', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,CO_NIVEL_ALFABETIZACAO,PC_AVALIADOS_LP
0,2025,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,65,68,78,67,70,72,74,76,78,80,4,91.53
1,2025,11,RO,1100023,Ariquemes,MUNICIPAL,62,66,80,65,68,71,73,76,78,80,4,70.07
2,2025,11,RO,1100031,Cabixi,MUNICIPAL,69,76,87,71,73,74,76,77,79,80,5,95.31
3,2025,11,RO,1100049,Cacoal,MUNICIPAL,63,66,85,65,68,71,73,76,78,80,5,91.84
4,2025,11,RO,1100056,Cerejeiras,MUNICIPAL,59,67,90,62,66,69,72,75,78,80,5,95.77


In [31]:
dic = pd.read_excel(
    caminho_xlsx, sheet_name="Variáveis", header=None,
    usecols=[1, 2], names=["campo", "descricao"]
).dropna(how="all")
display(dic)

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2023 (1),Percentual de alunos alfabetizados: percentual...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2024 (1),Percentual de alunos alfabetizados: percentual...
9,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2025 (1),Percentual de alunos alfabetizados: percentual...
10,META 2024 (2),Meta estabelecida para o município em 2024


In [36]:
caminho_xlsx = EXTRAIDOS_DIR / "2025" / "resultados_e_metas_ufs_2025_v1.xlsx"

# A planilha também possui duas linhas de cabeçalho.
# header=1 utiliza a segunda linha, que contém os códigos técnicos.
df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# Remove linhas do rodapé (observações)
df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")
print("\nColunas:", list(df_uf_meta.columns))

display(df_uf_meta.head())

Linhas : 27
Colunas: 16

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'PC_ALUNO_ALFABETIZADO_2025', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2025,12,AC,Acre,PÚBLICA,NaN,51,68,NaN,57,62,67,72,76,> 80,82
2,2025,27,AL,Alagoas,PÚBLICA,44,49,64,50,56,61,67,72,76,> 80,94
3,2025,13,AM,Amazonas,PÚBLICA,52,49,57,57,61,66,70,73,77,> 80,86
4,2025,16,AP,Amapá,PÚBLICA,42,47,60,48,54,60,66,71,76,> 80,88
5,2025,29,BA,Bahia,PÚBLICA,37,36,55,43,50,57,64,70,75,> 80,87


In [37]:
dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

display(dic_uf)

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
9,META 2024,Meta estabelecida para o estado ou Brasil em 2024
